# 03 — Re-sweep the three confirmatory cells on the repaired instrument

**Run only after notebook 01, and set `PROBE_TRAIN` below from its `recommended_config`.**

The existing stacks predate A8 and cannot support the confirmatory layer: they carry
(i) the broken row-shuffled control task, so `S ≈ R(real)` and the "dual gate" is a
single gate, and (ii) no random-projector floor, so H4 is a 512-d vs 128-d level
difference. This re-sweep fixes both and pins the calibrated probe-train size.

Realized grid, frozen at three cells by A7 §d: **color_strong, position_strong,
control_strong**. `scale_strong` and `orientation_strong` are excluded and live under
`results/probes_excluded/`.

~3 h per cell. Each cell is a separate resumable command; run what fits in a session,
Save Version, and continue.

**Setup:** Accelerator `GPU T4 x2`, Internet **On**. Attach the color / control /
position encoders as inputs.

## 1. Verify the GPU(s)

In [ ]:
!nvidia-smi

## 2. Clone the repo
Onto `/kaggle/working` (persists across restarts within a session).

In [ ]:
import os

REPO_URL = "https://github.com/chinesegorilla99/probe-capacity-invariance.git"
REPO_DIR = "/kaggle/working/probe-capacity-invariance"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

%cd {REPO_DIR}

## 3. Install dependencies
Without disturbing Kaggle's preinstalled, CUDA-matched `torch`/`torchvision`.

In [ ]:
!pip install -q -e . --no-deps
!pip install -q h5py

In [ ]:
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "| device count:", torch.cuda.device_count())

## 4. Download shapes3d + dsprites + build the image cache
`--build-cache` decompresses once into an uncompressed memmap the loaders mmap. Idempotent.

In [ ]:
!cd /kaggle/working/probe-capacity-invariance && python -m src.data.shapes3d --download --build-cache
!cd /kaggle/working/probe-capacity-invariance && python -m src.data.dsprites --download --build-cache

## 5. Restore checkpoints from a previous session (resume)
`Add Input -> a prior version's output` (or an uploaded encoder dataset), then run
this. It finds every `(?:color|control|position)_strong_seed*.pt` under `/kaggle/input` and restores it to
`results/encoders/<run_id>/` (`backbone*.pt` -> `backbone.pt`, `last_ckpt*.pt` ->
`last_ckpt.pt`). On a fresh first run there is nothing to restore. Only color/control/position
files are touched.

In [ ]:
import re, shutil
from pathlib import Path

REPO  = Path("/kaggle/working/probe-capacity-invariance")
ENC   = REPO / "results" / "encoders"
INPUT = Path("/kaggle/input")
_RID  = re.compile(r"(?:color|control|position)_strong_seed\d+")

def _target(name):
    n = name.lower()
    if "ckpt" in n:     return "last_ckpt.pt"
    if "backbone" in n: return "backbone.pt"
    return None

found = {}
for p in sorted(INPUT.rglob("*.pt")) if INPUT.exists() else []:
    m, tgt = _RID.search(p.as_posix()), _target(p.name)
    if m and tgt:
        found.setdefault((m.group(0), tgt), p)     # first match per (run_id, kind)

if not found:
    print("nothing to restore -- fresh start")
for (rid, tgt), src in sorted(found.items()):
    dst = ENC / rid / tgt
    dst.parent.mkdir(parents=True, exist_ok=True)
    if not dst.exists():
        shutil.copy2(src, dst)
    print(f"{rid:34s} {tgt:14s} <- {src}")

## 6. Integrity check — keep good checkpoints, purge corrupt ones
Each restored `.pt` is opened as a zip. A valid `backbone.pt` means the seed is
done and training skips it. A corrupt `backbone.pt` is deleted with its
`last_ckpt.pt` so the seed retrains; a valid `last_ckpt.pt` with no backbone lets
training resume mid-run.

In [ ]:
import zipfile
from pathlib import Path

ENC = Path("/kaggle/working/probe-capacity-invariance/results/encoders")

def _state(p):
    if not p.exists():
        return "missing"
    try:
        return None if zipfile.ZipFile(p).testzip() is None else "corrupt"
    except Exception as e:
        return f"not-a-zip ({e})"

for d in sorted(ENC.glob("(?:color|control|position)_strong_seed*")):
    bb, ck = d / "backbone.pt", d / "last_ckpt.pt"
    bstat = _state(bb)
    if bstat is None:
        print(f"{d.name:34s} backbone OK -> skip"); continue
    if bstat != "missing":
        bb.unlink(missing_ok=True); ck.unlink(missing_ok=True)
        print(f"{d.name:34s} backbone {bstat} -> purged, will retrain"); continue
    print(f"{d.name:34s} "
          + ("last_ckpt OK -> resume" if _state(ck) is None else "fresh start"))

In [ ]:
# Restore prior probe/calibration OUTPUTS (not checkpoints) so a timed-out
# session continues instead of recomputing.
import shutil
from pathlib import Path

REPO = Path("/kaggle/working/probe-capacity-invariance"); INPUT = Path("/kaggle/input")
restored = 0
for src in list(INPUT.glob("*/results")) + list(INPUT.glob("*/probe-capacity-invariance/results")):
    for f in src.rglob("*"):
        if f.is_file() and f.suffix in (".npz", ".json", ".jsonl"):
            dst = REPO / "results" / f.relative_to(src)
            if not dst.exists():
                dst.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(f, dst); restored += 1
print(f"restored {restored} prior result files")

## 7. Pin the calibrated probe-train size

In [ ]:
# From notebook 01's recommended_config. 40000 is the pre-A8 value (saturated).
PROBE_TRAIN = 40000     # <-- SET ME from results/calibration/calibration_shapes3d.json
SEEDS = "0 1 2 3 4 5 6 7 8 9 10 11"
print("probe-train pinned to", PROBE_TRAIN)

## 8. Cell 1/3 — color (Shapes3D)

In [ ]:
!cd /kaggle/working/probe-capacity-invariance && python -m src.probes.run_sweep \
    --config configs/probe/ladder.yaml \
    --dataset shapes3d --condition color --strength strong \
    --encoders results/encoders/color_strong_seed*/backbone.pt \
    --random-seed {SEEDS} --subsample {PROBE_TRAIN} \
    --device cuda --num-workers 2 --resume --out-root results/probes

## 9. Cell 2/3 — control-aug (Shapes3D), gate-exempt per A2

In [ ]:
!cd /kaggle/working/probe-capacity-invariance && python -m src.probes.run_sweep \
    --config configs/probe/ladder.yaml \
    --dataset shapes3d --condition control --strength strong \
    --encoders results/encoders/control_strong_seed*/backbone.pt \
    --random-seed {SEEDS} --subsample {PROBE_TRAIN} \
    --device cuda --num-workers 2 --resume --out-root results/probes

## 10. Cell 3/3 — position (dSprites)

In [ ]:
!cd /kaggle/working/probe-capacity-invariance && python -m src.probes.run_sweep \
    --config configs/probe/ladder.yaml \
    --dataset dsprites --condition position --strength strong \
    --encoders results/encoders/position_strong_seed*/backbone.pt \
    --random-seed {SEEDS} --subsample {PROBE_TRAIN} \
    --device cuda --num-workers 2 --resume --out-root results/probes

## 11. Verify the A8 repairs landed in every cell

In [ ]:
import numpy as np, json
from pathlib import Path
for cell in ("color_strong", "control_strong", "position_strong"):
    d = Path("/kaggle/working/probe-capacity-invariance") / "results/probes" / cell
    if not (d / "stacks.npz").exists():
        print(f"{cell:16s} NOT YET SWEPT"); continue
    z = np.load(d / "stacks.npz"); m = json.loads((d / "meta.json").read_text())
    assert "random_projector" in z.files, f"{cell}: A8 §d random-projector floor MISSING"
    assert m.get("encoder_ckpts"), f"{cell}: checkpoint provenance MISSING"
    print(f"{cell:16s} arrays={sorted(z.files)}  "
          f"gate={m['quality_gate']['n_passed']}/{m['quality_gate']['n_encoders']}  "
          f"n_ckpts={len(m['encoder_ckpts'])}  probe_train={m['probe_train_size']}")

In [ ]:
# --- persist for the next session --------------------------------------------
# /kaggle/working is the notebook's output. Click "Save Version" when this
# finishes, then Add Input -> this output on the next run to resume.
import shutil
from pathlib import Path
src = Path("/kaggle/working/probe-capacity-invariance/results"); dst = Path("/kaggle/working/results")
shutil.rmtree(dst, ignore_errors=True); shutil.copytree(src, dst)
print(f"persisted {sum(1 for _ in dst.rglob('*') if _.is_file())} files "
      f"-> click 'Save Version' now")